# Hospital Financial Distress Prediction
This notebook documents the process of building a model to predecit financial status for acute care facilities using 5 years of CMS Medicare Cost Reports (2019-2023)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, shap, warnings

# Machine Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

## Data Loading 
Five years of CMS Medicare Cost Reports (2019-2023) are stacked into a single dataframe with a year column added for context.

In [2]:
df = pd.concat([
    pd.read_csv(f'data/CostReport_{year}_Final.csv').assign(year=year)
    for year in range(2019, 2024)
], ignore_index=True)

print(f'Dataset Shape: {df.shape}')
df.head()

Dataset Shape: (30400, 118)


,rpt_rec_num,Provider CCN,Hospital Name,Street Address,City,State Code,Zip Code,County,Medicare CBSA Number,Rural Versus Urban,...,Total Other Income,Total Income,Total Other Expenses,Net Income,Cost To Charge Ratio,Net Revenue from Medicaid,Medicaid Charges,Net Revenue from Stand-Alone CHIP,Stand-Alone CHIP Charges,year
0,650479,201302,LINCOLNHEALTH,6 ST. ANDREWS LANE,BOOTHBAY HARBOR,ME,04538-,LINCOLN,99920.0,R,...,61805.0,-517955.0,NaN,-517955.0,0.594265,1692896.0,2070344.0,NaN,NaN,2019
1,655727,342017,ASHEVILLE SPECIALTY HOSPITAL LLC,428 BILTMORE AVENUE,ASHEVILLE,NC,28801,BUNCOMBE,11700.0,U,...,54959.0,1033596.0,NaN,1033596.0,NaN,NaN,NaN,NaN,NaN,2019
2,661241,344033,WALTER B JONES LAKESIDE,2577 WEST FIFTH STREET,GREENVILLE,NC,27834,PITT,24780.0,U,...,7847.0,-918671.0,NaN,-918671.0,1.827490,NaN,NaN,NaN,NaN,2019
3,661372,341316,HIGHLANDS CASHIERS HOSPITAL,HIGHWAY 64E,HIGHLANDS,NC,28741-,MACON,99934.0,R,...,2785450.0,1482484.0,NaN,1482484.0,0.557117,162586.0,470364.0,NaN,NaN,2019
4,662990,670087,SCOTT & WHITE CEDAR PARK,900 E WHITESTONE BLVD,CEDAR PARK,TX,78613,WILLIAMSON,12420.0,U,...,91176.0,-486013.0,NaN,-486013.0,0.257723,67094.0,909270.0,NaN,NaN,2019


## Facility Type Filter
Analysis is scoped to acute care hospital types: general short term (1), cancer (3), children's (7), and rural emergency hospitals (12). Long-term care, psychiatric, rehabilitation, religious non-medical, and other facility types operate under fundamentally different reimbursement models and length-of-stay profiles, making days-based operational ratios incomparable across types. Type 9 (Other) is excluded due to undefined facility classification.

In [3]:
# Evaluate distribution of facility types
df['Provider Type'].value_counts()

Provider Type
1     23099
4      3082
2      1771
5      1745
7       407
9       152
3        62
6        54
12       18
10        6
11        4
Name: count, dtype: int64

In [4]:
df = df[df['Provider Type'].isin([1, 3, 7, 12])]
df.shape

(23586, 118)

## Exploratory Data Analysis

In [5]:
# List of potential feature columns for model prediction. Features that are directly part of the target (Net Income) are eliminated as the goal is to determine if there are other non-financial factors that contribute to facility financial status

candidate_cols = [
    'Number of Beds', 'FTE - Employees on Payroll', 'Total Days Title V',
    'Total Days Title XVIII', 'Total Days Title XIX', 'Total Days (V + XVIII + XIX + Unknown)',
    'Total Bed Days Available', 'Total Discharges Title V', 'Total Discharges Title XVIII',
    'Total Discharges Title XIX', 'Total Discharges (V + XVIII + XIX + Unknown)', 
    'Cost of Charity Care', 'Cost of Uncompensated Care', 'Inpatient Total Charges',
    'Outpatient Total Charges', 'Combined Outpatient + Inpatient Total Charges',
    'Rural Versus Urban', 'Provider Type', 'Type of Control', 'CCN Facility Type', 'State Code', 'Net Income'
]

df[candidate_cols].describe().round(2)

,Number of Beds,FTE - Employees on Payroll,Total Days Title V,Total Days Title XVIII,Total Days Title XIX,Total Days (V + XVIII + XIX + Unknown),Total Bed Days Available,Total Discharges Title V,Total Discharges Title XVIII,Total Discharges Title XIX,Total Discharges (V + XVIII + XIX + Unknown),Cost of Charity Care,Cost of Uncompensated Care,Inpatient Total Charges,Outpatient Total Charges,Combined Outpatient + Inpatient Total Charges,Provider Type,Type of Control,Net Income
count,23351.00,23291.00,667.00,23161.00,21188.00,23309.00,23351.00,623.00,23195.00,20814.00,23304.00,2.129200e+04,2.246700e+04,2.336400e+04,2.336100e+04,2.340300e+04,23586.00,23586.00,2.338000e+04
mean,226.42,1040.06,2313.04,8798.19,3619.83,33996.60,53444.07,439.95,1633.93,629.10,6588.12,6.614668e+06,9.011397e+06,4.958250e+08,4.964171e+08,9.905249e+08,1.12,4.02,1.786298e+07
std,10487.35,1864.89,5301.49,13720.24,8173.41,55226.47,70790.20,694.46,2358.21,1358.61,9833.23,2.020080e+07,2.367870e+07,1.058986e+09,9.044303e+08,1.901446e+09,0.84,3.38,1.117910e+08
min,1.00,0.01,1.00,1.00,1.00,1.00,26.00,1.00,1.00,1.00,1.00,1.000000e+00,-1.569430e+05,1.000000e+00,7.000000e+01,1.000000e+00,1.00,1.00,-3.955820e+09
25%,25.00,154.52,99.50,1007.00,114.00,2638.00,9125.00,14.00,158.00,26.00,455.00,3.468900e+05,1.037286e+06,1.308695e+07,4.988978e+07,6.723059e+07,1.00,2.00,-1.204512e+06
50%,70.00,414.56,607.00,2812.00,751.00,10085.00,24820.00,150.00,594.00,147.00,2418.00,1.632076e+06,2.963322e+06,9.348238e+07,1.943806e+08,2.951533e+08,1.00,2.00,3.105010e+06
75%,201.00,1160.29,2520.50,11364.00,3356.00,44625.00,72635.00,585.50,2268.00,609.75,9335.50,5.748094e+06,8.460310e+06,5.306452e+08,6.092785e+08,1.156548e+09,1.00,4.00,1.779454e+07
max,1594784.00,33793.00,67185.00,208248.00,139073.00,911866.00,1040250.00,7148.00,31284.00,19799.00,168398.00,8.192046e+08,8.504827e+08,2.447432e+10,2.337112e+10,4.511504e+10,12.00,13.00,7.919570e+09


In [6]:
# At the time of this project published public information states New York Presbyterian Weill Cornell Medical Center is the largest hospital by bed count with 2,850 beds
df[df['Number of Beds'] > 3000][['Hospital Name', 'City', 'year', 'Provider Type', 'FTE - Employees on Payroll', 'Number of Beds', 'Total Bed Days Available']].sort_values(by='Hospital Name')

,Hospital Name,City,year,Provider Type,FTE - Employees on Payroll,Number of Beds,Total Bed Days Available
728,BEAR LAKE MEMORIAL HOSPITAL,MONTPELIER,2019,1,134.62,52913.0,7665.0
7582,MERCY HOSPITAL & MEDICAL CENTER,CHICAGO,2020,1,958.07,144837.0,61437.0
4404,OKLAHOMA HEART HOSPITAL,OKLAHOMA CITY,2019,1,1564.52,28401.0,36135.0
10337,WAYNE HOSPITAL COMPANY,GREENVILLE,2020,1,371.00,1594784.0,14640.0


In [7]:
# Checking for data on the hospitals with unusually high bed counts in other years to verify outlier status
bed_outlier_ccns = df[df['Number of Beds'] > 3000]['Provider CCN'].tolist()
df[df['Provider CCN'].isin(bed_outlier_ccns)][['Hospital Name', 'year', 'Number of Beds', 'Total Bed Days Available']].sort_values(by=['Hospital Name', 'year'])

,Hospital Name,year,Number of Beds,Total Bed Days Available
728,BEAR LAKE MEMORIAL HOSPITAL,2019,52913.0,7665.0
6274,BEAR LAKE MEMORIAL HOSPITAL,2020,21.0,7686.0
12408,BEAR LAKE MEMORIAL HOSPITAL,2021,21.0,7665.0
18895,BEAR LAKE MEMORIAL HOSPITAL,2022,21.0,7665.0
25311,BEAR LAKE MEMORIAL HOSPITAL,2023,21.0,7665.0
15974,INSIGHTS HOSPITAL AND MEDICAL CENTER,2021,52.0,11180.0
20040,INSIGHTS HOSPITAL AND MEDICAL CENTER,2022,52.0,18980.0
26552,INSIGHTS HOSPITAL AND MEDICAL CENTER,2023,52.0,18980.0
2895,MERCY HOSPITAL & MEDICAL CENTER,2019,203.0,74095.0
7582,MERCY HOSPITAL & MEDICAL CENTER,2020,144837.0,61437.0


In [8]:
# Removing rows with confirmed single year reporting errors. 
df = df[df['Number of Beds'] <= 3000]
df.shape

(23347, 118)

In [9]:
df[df['FTE - Employees on Payroll'] > 15000][['Hospital Name', 'City', 'year', 'Provider Type', 'FTE - Employees on Payroll', 'Number of Beds', 'Total Bed Days Available']].sort_values(by='Hospital Name')

,Hospital Name,City,year,Provider Type,FTE - Employees on Payroll,Number of Beds,Total Bed Days Available
16300,ADVENTHEALTH ORLANDO,ORLANDO,2021,1,19277.00,2791.0,1018715.0
29633,ADVENTHEALTH ORLANDO,ORLANDO,2023,1,25192.00,2787.0,1017255.0
2985,ADVENTHEALTH ORLANDO,ORLANDO,2019,1,18864.83,2735.0,998275.0
23412,ADVENTHEALTH ORLANDO,ORLANDO,2022,1,22858.00,2738.0,999370.0
7538,ADVENTHEALTH ORLANDO,ORLANDO,2020,1,19457.00,2812.0,1029192.0
...,...,...,...,...,...,...,...
23715,VANDERBILT UNIVERSITY MEDICAL CENTER,NASHVILLE,2022,1,25087.17,1084.0,388920.0
9704,VANDERBILT UNIVERSITY MEDICAL CENTER,NASHVILLE,2020,1,24119.10,1056.0,385440.0
2361,VANDERBILT UNIVERSITY MEDICAL CENTER,NASHVILLE,2019,1,22422.76,1056.0,365595.0
17009,VANDERBILT UNIVERSITY MEDICAL CENTER,NASHVILLE,2021,1,23247.37,1056.0,385440.0
